In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import matplotlib.pyplot as plt

DATA_DIR = Path.home() / "kkbox-churn" / "data" / "parquet"
MODEL_DIR = Path.home() / "kkbox-churn" / "models"

master = pd.read_parquet(DATA_DIR / "master_dataset.parquet")

# Load global model
drop_cols = [
    'msno', 'last_transaction_date', 'last_expire_date',
    'last_listen_date', 'first_listen_date', 'cluster_k5'
]
master_model = master.drop(columns=drop_cols)
for col in ['city', 'registered_via', 'bd_bucket', 'auto_renew_switch']:
    master_model[col] = master_model[col].astype('category')

X = master_model.drop(columns=['is_churn', 'segment'])
final_model = joblib.load(MODEL_DIR / "lgbm_churn_final.pkl")

# Get churn probabilities for all 970K users
master['churn_prob'] = final_model.predict_proba(X)[:, 1]

print(f"Master shape: {master.shape}")
print(f"Churn prob distribution:")
print(master['churn_prob'].describe())

Master shape: (970960, 110)
Churn prob distribution:
count    970960.000000
mean          0.158043
std           0.312335
min           0.000021
25%           0.000403
50%           0.005394
75%           0.097233
max           0.999340
Name: churn_prob, dtype: float64


In [2]:
# Cell 2 — CLV calculation
# CLV formula: avg_monthly_revenue / churn_probability
# Interpretation: expected total revenue before customer churns

# Monthly revenue proxy — use avg_amount_paid / avg_plan_days * 30
master['monthly_revenue'] = (
    master['avg_amount_paid'] / master['avg_plan_days'].replace(0, np.nan) * 30
).fillna(master['avg_amount_paid'])

# Clip churn probability to avoid division by near-zero
# Min 0.1% churn = max 1000 months CLV (reasonable ceiling)
master['churn_prob_clipped'] = master['churn_prob'].clip(lower=0.001)

# CLV = monthly_revenue / churn_probability
master['clv'] = master['monthly_revenue'] / master['churn_prob_clipped']

# Cap CLV at 99th percentile to handle outliers
clv_cap = master['clv'].quantile(0.99)
master['clv'] = master['clv'].clip(upper=clv_cap)

print(f"CLV distribution (TWD):")
print(master['clv'].describe().round(2))

print(f"\nMonthly revenue distribution (TWD):")
print(master['monthly_revenue'].describe().round(2))

CLV distribution (TWD):
count    970960.00
mean      58332.04
std       61065.34
min           0.00
25%        1539.11
50%       24831.72
75%       99000.00
max      201588.24
Name: clv, dtype: float64

Monthly revenue distribution (TWD):
count    970960.00
mean        133.70
std          31.68
min           0.00
25%          99.00
50%         141.01
75%         154.14
max        1968.00
Name: monthly_revenue, dtype: float64


In [3]:
# Cell 3 — CLV by segment + revenue at risk
clv_by_segment = master.groupby('segment').agg(
    users=('msno', 'count'),
    avg_clv=('clv', 'mean'),
    median_clv=('clv', 'median'),
    total_clv=('clv', 'sum'),
    avg_churn_prob=('churn_prob', 'mean'),
    avg_monthly_revenue=('monthly_revenue', 'mean'),
    churn_rate=('is_churn', 'mean')
).round(2)

clv_by_segment['revenue_at_risk'] = (
    clv_by_segment['total_clv'] * clv_by_segment['avg_churn_prob']
).round(2)

clv_by_segment = clv_by_segment.sort_values('avg_clv', ascending=False)

print("CLV by segment (TWD):")
print(clv_by_segment.to_string())

print(f"\nTotal revenue at risk: {clv_by_segment['revenue_at_risk'].sum():,.0f} TWD")
print(f"Total CLV across all users: {clv_by_segment['total_clv'].sum():,.0f} TWD")

CLV by segment (TWD):
                 users   avg_clv  median_clv     total_clv  avg_churn_prob  avg_monthly_revenue  churn_rate  revenue_at_risk
segment                                                                                                                     
At-Risk         438200  68005.22    99000.00  2.979989e+10            0.13               112.75        0.09     3.873985e+09
New_Reengaged    22416  60934.51    54831.89  1.365908e+09            0.14               131.32        0.07     1.912271e+08
Champion        397304  52479.09     8096.45  2.085015e+10            0.15               154.33        0.06     3.127523e+09
Power_Listener   92479  49818.65     9946.24  4.607179e+09            0.16               145.90        0.06     7.371486e+08
Lost             20561    727.35      134.41  1.495512e+07            0.86               129.05        0.82     1.286140e+07

Total revenue at risk: 7,942,745,410 TWD
Total CLV across all users: 56,638,082,238 TWD


In [5]:
# Cell 4 — Diagnose churn prob by segment
print(master.groupby('segment')[['churn_prob', 'is_churn']].describe().round(4))

               churn_prob                                                  \
                    count    mean     std     min     25%     50%     75%   
segment                                                                     
At-Risk          438200.0  0.1347  0.3157  0.0000  0.0003  0.0005  0.0118   
Champion         397304.0  0.1478  0.2703  0.0000  0.0009  0.0183  0.1455   
Lost              20561.0  0.8612  0.3036  0.0006  0.9778  0.9914  0.9975   
New_Reengaged     22416.0  0.1367  0.2956  0.0000  0.0004  0.0020  0.0472   
Power_Listener    92479.0  0.1616  0.2888  0.0000  0.0006  0.0148  0.1880   

                        is_churn                                           
                   max     count    mean     std  min  25%  50%  75%  max  
segment                                                                    
At-Risk         0.9990  438200.0  0.0891  0.2850  0.0  0.0  0.0  0.0  1.0  
Champion        0.9993  397304.0  0.0623  0.2418  0.0  0.0  0.0  0.0  1.0  
Los

In [6]:
# Cell 4 — Relabel segments based on actual churn behavior
label_remap = {
    'At-Risk': 'Short_Cycle',      # expiring soon but mostly auto-renewing
    'Champion': 'Mid_Value',        # moderate engagement, moderate risk
    'Power_Listener': 'High_Engage',# heavy listeners, moderate risk
    'New_Reengaged': 'Returning',   # came back recently
    'Lost': 'Lost'                  # genuinely lost, 86% churn prob
}

master['segment_v2'] = master['segment'].map(label_remap)

print(master.groupby('segment_v2').agg(
    users=('msno', 'count'),
    median_churn_prob=('churn_prob', 'median'),
    actual_churn_rate=('is_churn', 'mean'),
    avg_clv=('clv', 'mean')
).sort_values('median_churn_prob', ascending=False).round(4).to_string())

              users  median_churn_prob  actual_churn_rate     avg_clv
segment_v2                                                           
Lost          20561             0.9914             0.8195    727.3537
Mid_Value    397304             0.0183             0.0623  52479.0891
High_Engage   92479             0.0148             0.0561  49818.6480
Returning     22416             0.0020             0.0651  60934.5108
Short_Cycle  438200             0.0005             0.0891  68005.2222


In [7]:
# Cell 5 — Save CLV + retention priority matrix
master['segment'] = master['segment_v2']
master = master.drop(columns=['segment_v2'])

# Retention priority = high churn prob + high CLV = top priority
master['retention_priority'] = (
    master['churn_prob'] * master['clv']
).rank(ascending=False, method='dense').astype(int)

# Save updated master
master.to_parquet(DATA_DIR / "master_dataset.parquet", index=False)

# Retention strategy matrix
print("\n=== RETENTION STRATEGY MATRIX ===\n")
strategy = {
    'Lost':         'No action — write off. CAC > CLV.',
    'Mid_Value':    'Targeted discount offer — 1 month free or price lock.',
    'High_Engage':  'Premium upsell — offer hi-fi audio or offline downloads.',
    'Returning':    'Engagement campaign — playlist recommendations, social features.',
    'Short_Cycle':  'No action needed — auto-renewing, highest CLV.'
}

for seg, action in strategy.items():
    users = master[master['segment'] == seg].shape[0]
    avg_clv = master[master['segment'] == seg]['clv'].mean()
    churn = master[master['segment'] == seg]['churn_prob'].median()
    print(f"{seg:<15} | Users: {users:>7,} | Median churn: {churn:.4f} | Avg CLV: {avg_clv:>10,.0f} TWD")
    print(f"               Action: {action}\n")

print(f"Saved master with CLV and retention priority.")


=== RETENTION STRATEGY MATRIX ===

Lost            | Users:  20,561 | Median churn: 0.9914 | Avg CLV:        727 TWD
               Action: No action — write off. CAC > CLV.

Mid_Value       | Users: 397,304 | Median churn: 0.0183 | Avg CLV:     52,479 TWD
               Action: Targeted discount offer — 1 month free or price lock.

High_Engage     | Users:  92,479 | Median churn: 0.0148 | Avg CLV:     49,819 TWD
               Action: Premium upsell — offer hi-fi audio or offline downloads.

Returning       | Users:  22,416 | Median churn: 0.0020 | Avg CLV:     60,935 TWD
               Action: Engagement campaign — playlist recommendations, social features.

Short_Cycle     | Users: 438,200 | Median churn: 0.0005 | Avg CLV:     68,005 TWD
               Action: No action needed — auto-renewing, highest CLV.

Saved master with CLV and retention priority.
